### Setup datasets

### NOTES

You need to have the smol dataset on your google drive in MyDrive/smol.

In [34]:
# CONFIG
LANGUAGES = ["es", "lij", "mfe"]

USE_SMOLDOCS = False # May give better results for low-resource languages mBART-50 doesn't know, but not all languages have docs, check documentation.

EPOCHS = 3 # Currently this gives the best results for the time invested.

In [35]:
# COLLAB
from google.colab import drive
drive.mount('/content/drive')

# INSTALLS
!pip install evaluate
!pip install sacrebleu

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
# IMPORTS
smol_path = "/content/drive/MyDrive/smol"

import time
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, concatenate_datasets
from transformers import MBartForConditionalGeneration, MBart50Tokenizer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
import evaluate
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
import os


In [37]:
# ===================== HELPER FUNCTIONS =====================================#

# To figure out what languages have smoldocs
def try_load_jsonl(path):
    if os.path.exists(path):
        return load_dataset("json", data_files=path)["train"]
    return None

# Load all the datasets
def load_language_datasets(lang):
    gatitos = try_load_jsonl(f"{smol_path}/gatitos/en_{lang}.jsonl")
    smolsent = try_load_jsonl(f"{smol_path}/smolsent/en_{lang}.jsonl")
    smoldoc = None

    if USE_SMOLDOCS:
        smoldoc = try_load_jsonl(f"{smol_path}/smoldoc/en_{lang}.jsonl")

    # Normalize gatitos ("trgs" -> "trg")
    if gatitos is not None:
        gatitos = gatitos.map(unify_gatitos)

    # Flatten smoldoc to smolsent
    if smoldoc is not None:
        smoldoc_flat = smoldoc.map(flatten_smoldoc_to_smolsent, batched=True, remove_columns=smoldoc.column_names)
        smolsent = concatenate_datasets([smolsent, smoldoc_flat])

    return gatitos, smolsent

# Convert 'trgs' list -> single 'trg' string for gatitos
def unify_gatitos(example):
    example["trg"] = example["trgs"][0]
    return example

# Convert smoldoc 'srcs' and 'trgs' into 'src' and 'trg'
def flatten_smoldoc_to_smolsent(batch):
    new_src = []
    new_trg = []
    new_tl  = []

    for srcs, trgs, tl in zip(batch["srcs"], batch["trgs"], batch["tl"]):

        # ensure srcs/trgs match
        if len(srcs) != len(trgs):
            print("MISMATCH BETWEEN TRGS AND SRC")

        for s, t in zip(srcs, trgs):
            new_src.append(s)
            new_trg.append(t)
            new_tl.append(tl)

    if len(new_src) > 0:
        print(f"Total entries: {len(new_src)}")
        print("First flattened entry:")
        print(f"  src: {new_src[1]}")
        print(f"  trg: {new_trg[1]}")
        print(f"  tl:  {new_tl[1]}")
        print("-" * 50)

    return {
        "src": new_src,
        "trg": new_trg,
        "tl":  new_tl
    }

# Create the train, test, and val split
def split_smolsent(ds):
    train_test = ds.train_test_split(test_size=0.30, seed=42)
    train_ds = train_test["train"]
    test_val_ds = train_test["test"]

    val_test = test_val_ds.train_test_split(test_size=2/3, seed=42)
    val_ds = val_test["train"]
    test_ds = val_test["test"]

    return train_ds, val_ds, test_ds

# Create the special language token corresponding to a language
def create_language_ID(lang):
    return f"{lang}_XX"

In [38]:
#========================== MAIN FUNCTION LOOP ===============================#

# Temporary structures
training_sets = []
gatitos_sets = []
val_sets = []

# The language key mapped to test set
test_dataset = {}

for lang in LANGUAGES:
  print(f"Processing language: {lang}")

  gatitos, smolsent = load_language_datasets(lang)
  gatitos_sets.append(gatitos)

  train, val, test = split_smolsent(smolsent)

  training_sets.append(train)
  val_sets.append(val)

  test_dataset[lang] = test

# Merge sets into final
train_dataset = concatenate_datasets(training_sets + gatitos_sets)
val_dataset = concatenate_datasets(val_sets)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test sizes:", {k: len(v) for k, v in test_dataset.items()})

Processing language: es


Map:   0%|          | 0/3991 [00:00<?, ? examples/s]

Processing language: lij


Map:   0%|          | 0/3976 [00:00<?, ? examples/s]

Processing language: mfe


Map:   0%|          | 0/3997 [00:00<?, ? examples/s]

Train size: 13776
Validation size: 258
Test sizes: {'es': 173, 'lij': 173, 'mfe': 173}


In [32]:
# Load model
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# Setup language tokens that don't exist in mBART
special_tokens = [f"<{lang}_XX>" for lang in LANGUAGES]
tokenizer.add_tokens(special_tokens, special_tokens=True)
model.resize_token_embeddings(len(tokenizer))

# Map language "es" to token ID "es_XX"
LANG_MAP = {lang: create_language_ID(lang) for lang in LANGUAGES}
print(LANG_MAP)

# Detect GPU or use CPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model.to(device)

{'es': 'es_XX', 'lij': 'lij_XX', 'mfe': 'mfe_XX'}
Using device: cpu


MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250057, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250057, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
def evaluate_translation_model(model, tokenizer, test_datasets, batch_size=16):
    """
    Evaluate mBART on multiple languages.

    Args:
        model: HuggingFace mBART model
        tokenizer: HuggingFace tokenizer
        test_datasets: dict of {lang_code: Dataset}
        batch_size: batch size for DataLoader

    Returns:
        results: dict of {lang_code: BLEU score}
    """
    bleu_metric = evaluate.load("sacrebleu")
    chrf_metric = evaluate.load("chrf")
    model.eval()
    results = {}

    for lang, dataset in test_datasets.items():
        print(f"Evaluating language: {lang}")
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        preds = []
        refs  = []

        for batch in tqdm(loader):
            src_texts = batch["src"]
            tgt_texts = batch["trg"]

            # tokenize source
            inputs = tokenizer(src_texts, return_tensors="pt", truncation=True, padding=True, max_length=128).to(model.device)

            # Determine forced_bos_token_id
            tgt_lang = LANG_MAP[lang]
            if tgt_lang in tokenizer.lang_code_to_id:
                bos_id = tokenizer.lang_code_to_id[tgt_lang]
            else:
                bos_id = tokenizer.convert_tokens_to_ids(f"<{tgt_lang}>")

            # generate translations
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    forced_bos_token_id=bos_id,
                    max_length=128
                )

            # decode predictions
            decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            decoded_preds = [p.strip() for p in decoded_preds]

            preds.extend(decoded_preds)
            refs.extend([[t.strip()] for t in tgt_texts])

        # compute BLEU
        bleu = bleu_metric.compute(predictions=preds, references=refs)
        chrf = chrf_metric.compute(predictions=preds, references=refs)

        print(f"{lang} BLEU: {bleu['score']:.2f}, chrF: {chrf['score']:.2f}")
        results[lang] = {"bleu": bleu["score"], "chrf": chrf["score"]}

    return results

# Evaluate all languages
bleu_scores = evaluate_translation_model(model, tokenizer, test_dataset)

In [ ]:
# TRAINING

def preprocess_function(batch):

    # Tokenize source (English)
    model_inputs = tokenizer(
        batch["src"],
        max_length=128,
        truncation=True
    )

    # Map tl -> BOS
    bos_ids = []
    for tl in batch["tl"]:
        lang = LANG_MAP[tl]
        bos_token = f"<{lang}>"
        bos_ids.append(tokenizer.convert_tokens_to_ids(bos_token))

    # Tokenize target normally (We can't tokenize like the input because mBART might not know language tokenizer... Also we don't need <s> tokens and so on.
    target_ids = [tokenizer.encode(t, truncation=True, max_length=124, add_special_tokens=False) for t in batch["trg"]]

    # Prepend BOS
    labels = []
    for bos, seq in zip(bos_ids, target_ids):
        new_seq = [bos] + seq + [tokenizer.eos_token_id]
        labels.append(new_seq)

    model_inputs["labels"] = labels

    return model_inputs

# tokenize everything
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val   = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

# Collator for some padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Args
#output_dir_name = "finetuned-mBart-ES"
training_args = Seq2SeqTrainingArguments(
    output_dir="finetuned-mBart-ES",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    learning_rate=3e-5,

    label_smoothing_factor=0.1,
    save_strategy="epoch",
    predict_with_generate=True,
    logging_strategy="steps",
    logging_steps=20,

    warmup_ratio=0.1 # May be removed?

    report_to="none"
)

# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train!
trainer.train()


Map:   0%|          | 0/13776 [00:00<?, ? examples/s]

Map:   0%|          | 0/258 [00:00<?, ? examples/s]

/tmp/ipython-input-1392051387.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
20,8.855600
40,8.452400
60,7.509500
80,6.754300
100,6.567400
120,6.235300
140,6.032000
160,5.772800
180,5.804700
200,5.385000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=2583, training_loss=3.5517095661052567, metrics={'train_runtime': 2507.6069, 'train_samples_per_second': 16.481, 'train_steps_per_second': 1.03, 'total_flos': 2066662986350592.0, 'train_loss': 3.5517095661052567, 'epoch': 3.0})

In [ ]:
# Test the model again
bleu_scores = evaluate_translation_model(model, tokenizer, test_dataset)

Evaluating language: es


100%|██████████| 11/11 [00:24<00:00,  2.24s/it]


es BLEU: 33.84, chrF: 60.32
Evaluating language: lij


100%|██████████| 11/11 [00:37<00:00,  3.45s/it]


lij BLEU: 11.83, chrF: 40.11
Evaluating language: mfe


100%|██████████| 11/11 [00:27<00:00,  2.46s/it]

mfe BLEU: 14.26, chrF: 44.33
